# TL;DR. 
Below we instantiate and run inference on Mistral LLMs, mainly using the `transformers` HF library.  

Other client APIs like OpenAI Python SDK or the OpenAI REST API are included for models,  

and we can use them as long as a local vLLM web server is up and running  

__NOTE!__ The vLLM server for `mistralai/Magistral-small-2506` does not work.  

----

# `mistral-7b-instruct-v0.1`

## Inference with `transformers.AutoModelForCausalLM`

The model card is [here](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1).  

This is a __gated__ model, meaning we need to logon to HF to download it (as soon as we have the model locally of course we need no other credentials).  

The model card uses Mistral's Python library `mistral_common` for tokenization. This is NOT mandatory however since a `LlamaTokenizerFast` can be instantiated with
```python
AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
```

So, in the cells below I use my `ModelConfig` model to instantiate the LLM from my local model cache.  

Note that the `tokenizer_config.json` file in the HF repo (in `~/.cache/huggingface/hub/model--mistralai--Mistral-7B-Instruct-v0.1/snapshots/<revision>/tokenizer_config.json`)   

contains at the bottom a `jinja2` fragment with the tokenizer's `chat_template`. This is used for converting the `List[Dict[role, content]]` chat messages to a string annotated with Mistral's special tokens.  



In [1]:
import os

from huggingface_hub import login

import util as ut

SETTINGS = ut.Settings.make(env_path="../.env")
os.environ["HF_TOKEN"] = SETTINGS.hf_finegrained_token
os.environ["DISABLE_IMPLICIT_TOKEN"] = "1"
model = ut.ModelConfig.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
messages = [
    {"role": "user", "content": "What is your favourite condiment?"},
    {"role": "assistant", "content": "Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!"},
    {"role": "user", "content": "Do you have mayonnaise recipes?"}
]

### Tokenization
Below we hack (at last) the Mistral message tokenization.  

This is important as it permits to write custom converters for chat inputs that do not comply with the OpenAI chat format.

In [ ]:
import os
from transformers.models.llama.tokenization_llama_fast import LlamaTokenizerFast

# apply_chat_template will use the chat_template of the model, in 
# ~/.cache/huggingface/hub/model--mistralai--Mistral-7B-Instruct-v0.1/snapshots/<commit-hash>/tokenizer_config.json (scroll to the bottom to see th jinja configuration),
# to convert the OpenAI chat format to a mere string.
annotated_ids = model.tokenizer.apply_chat_template(messages, return_tensors="pt")

# annotated_ids is a 1xN Tensor of token ids of the final string. How do we obtain the string?
# The tokenizer of the util.ModelConfig object is a LlamaTokenizerFast class
print("model.tokenizer type is LlamaTokenizerFast:", isinstance(model.tokenizer, LlamaTokenizerFast))

# ...we can therefore decode the real prompt to the LLM
print(model.tokenizer.decode(annotated_ids[0]))

model.tokenizer type is LlamaTokenizerFast: True
<s> [INST] What is your favourite condiment? [/INST] Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!</s> [INST] Do you have mayonnaise recipes? [/INST]


In [ ]:
model.generate(input_ids=annotated_ids, max_new_tokens=1000)

## Inference with OpenAI Python SDK

In [62]:
from openai import OpenAI
# client = OpenAI(api_key=os.environ["HF_FINEGRAINED_TOKEN"], base_url="http://localhost:8000/v1")
client = OpenAI(api_key="abc-123", base_url="http://localhost:8000/v1")

query = "Write 4 sentences, each with at least 8 words. Make sure that every sentence has exactly one word less than its previous one."
messages = [
    # {"role": "system", "content": "User messages are instructions that you must follow exactly."},
    {"role": "user", "content": query}
]

responses = client.chat.completions.create(
  model=model,
  messages=messages,
  stream=True,
  temperature=0.7,
  top_p=0.95,
  max_tokens=4096,
  timeout=50
)

for chunk in responses:
  if chunk.created is True: 
    print("Created")
  print(chunk.choices[0].delta.content, end="", flush=True)

 1. The big brown dog chased the little red cat.
2. The tall man walked the short woman.
3. The swift cheetah leaped the wide river.
4. The long black snake slithered the narrow path.

# `magistral-small-2506`

1. According to the [model card](https://huggingface.co/mistralai/Magistral-Small-2506) we can run inference only using a `vLLM` server.  

   I could not make the model work on my Mac however! The vLLM server, started with the parameters proposed in the model card

   consumes too much memory and hangs when a request is sent by the OpenAI SDK client below.

2. There's no support for transformers (not mentioned in the model card).

## Inference on vLLM with OpenAI Py SDK 

$\textsf{\textcolor{red}{(Does not work with Magistral-small-2506)}}$

The code here assumes a local vllm server is started with the options used in [Magistral's model card on Huggingface](https://huggingface.co/mistralai/Magistral-Small-2506#vllm):  

```
vllm serve \
mistralai/Magistral-Small-2506 \
--tokenizer_mode mistral \
--config_format mistral \
--load_format mistral \
--tool-call-parser mistral \
--enable-auto-tool-choice \
--tensor-parallel-size 2
```

In [30]:
from huggingface_hub import hf_hub_download

def load_system_prompt(repo_id: str, filename: str) -> str:
    file_path = hf_hub_download(repo_id=repo_id, filename=filename)
    with open(file_path, "r") as file:
        system_prompt = file.read()
    return system_prompt

In [ ]:
model = "mistralai/Magistral-Small-2506"
SYSTEM_PROMPT = load_system_prompt("mistralai/Magistral-Small-2506", "SYSTEM_PROMPT.txt")
print(pattern_split(SYSTEM_PROMPT))

model = "mistralai/Mistral-7B-Instruct-v0.1"

In [58]:
from openai import OpenAI
# client = OpenAI(api_key=os.environ["HF_FINEGRAINED_TOKEN"], base_url="http://localhost:8000/v1")
client = OpenAI(api_key="abc-123", base_url="http://localhost:8000/v1")

query = "Write 4 sentences, each with at least 8 words. Now make absolutely sure that every sentence has exactly one word less than the previous sentence."
system = SYSTEM_PROMPT
messages = [
    {"role": "system", "content": "User messages are instructions that you must follow exactly"},
    {"role": "user", "content": query}
]

responses = client.chat.completions.create(
  model=model,
  messages=messages,
  stream=True,
  temperature=0.7,
  top_p=0.95,
  max_tokens=4096,
  timeout=50
)

for chunk in responses:
  if chunk.created is True: 
    print("Created")
  print(chunk.choices[0].delta.content, end="", flush=True)

 1. User messages are instructions that you must follow exactly.
2. They provide a clear and concise way to communicate with the system.
3. It is important to understand and comply with these messages to avoid any errors.
4. Follow user messages to ensure a successful interaction with the system.